# Container Resource Prediction — Full Pipeline
## Phase 1: Data Preprocessing + Sequence Generation
## Phase 2: GRU Model Architecture & Data Pipeline

---

### Pipeline Overview
```
Phase 1 — Preprocessing
  Step 1+2 : Load raw CSVs from Google Drive  →  Pivot long to wide format
  Step 3a  : Chronological split (60/20/20)   →  Normalize (train stats only)
  Step 3b  : Feature engineering              →  Lag diffs + rolling features
  Step 4   : Sliding window sequences         →  .npy files per horizon

Phase 2 — GRU Model Architecture
  Step 5   : SequenceDataset                  →  Wraps .npy as PyTorch Dataset
  Step 6   : DataLoaders                      →  Configurable batching
  Step 7   : GRUModel                         →  2-layer GRU -> FC -> Output
  Step 8   : Smoke test                       →  Verify shapes with real data
```

---
**Author:** Team-Dracasys | **Version:** Phase 1 + Phase 2 Combined

# STEP 0: Memory Monitoring Setup

In [ ]:
import psutil
import os
import gc

def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def log_memory(label: str):
    """Log memory usage with label."""
    gc.collect()  # Force garbage collection
    mem = get_memory_usage()
    print(f"💾 [{label}] RAM: {mem:.1f} MB")
    return mem

print("✓ Memory monitoring setup complete")
initial_mem = log_memory("Initial")

# STEP 1: Mount Google Drive & Verify Access

In [ ]:
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("⚠ Running locally (not Google Colab)")

print(f"Python version: {sys.version}")

In [ ]:
if IN_COLAB:
    from google.colab import drive
    from pathlib import Path
    
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    print("\n✓ Google Drive mounted!")
    
    raw_path = Path('/content/drive/My Drive/raw')
    
    if raw_path.exists():
        print(f"✓ Found raw folder at: {raw_path}")
        all_csv_files = list(raw_path.glob('**/*.csv'))
        print(f"✓ Found {len(all_csv_files)} CSV files")
    else:
        print(f"❌ raw folder not found at: {raw_path}")
else:
    print("To use this notebook, please run it in Google Colab")
    print("Open in Colab: https://colab.research.google.com")

# STEP 2: Setup Imports & Configure Paths

In [ ]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from typing import List, Dict, Tuple
import json
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ All imports successful")

In [ ]:
# Setup Paths
gd_raw_path    = Path('/content/drive/My Drive/raw')

# Intermediate files (CSV) stay in /content/ — small enough
output_base    = Path('/content/processed_data')
processed_path = output_base / 'processed'
merged_path    = output_base / 'merged'

# Sequences written DIRECTLY to Drive — too large for /content/ disk
sequences_path = Path('/content/drive/My Drive/processed_data/sequences')
sequences_path.mkdir(parents=True, exist_ok=True)

print("Path Configuration:")
print(f"  Raw input       : {gd_raw_path}")
print(f"  Merged CSVs     : {merged_path}  (local /content/)")
print(f"  Sequences       : {sequences_path}  (Google Drive - persists)")

log_memory("After setup")

# STEP 1 + 2: Load Raw Data & Pivot to Wide Format

In [ ]:
def load_data_from_drive_optimized(raw_path: Path) -> pd.DataFrame:
    """Load CSV files and pivot from long format to wide format.

    Raw CSV format (long):
        timestamp | cmdb_id | kpi_name | value

    Output format (wide):
        timestamp | cmdb_id | new_container_id | metric_1 | metric_2 | ...
    """
    logger.info(f"Loading data from Google Drive (RAM-optimized)...")

    csv_files = sorted(raw_path.glob('**/*.csv'))
    logger.info(f"Found {len(csv_files)} CSV files")

    if not csv_files:
        logger.error("No CSV files found!")
        return None

    # Target metrics to keep
    target_metrics = [
        'container_cpu_usage_seconds_total',
        'container_cpu_system_seconds_total',
        'container_cpu_user_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss',
        'container_memory_cache'
    ]

    all_dfs = []
    for idx, csv_file in enumerate(csv_files, 1):
        try:
            df = pd.read_csv(csv_file)
            # Keep only rows for target metrics
            if 'kpi_name' in df.columns:
                df = df[df['kpi_name'].isin(target_metrics)]
            all_dfs.append(df)
            if idx % 10 == 0:
                logger.info(f"  Loaded {idx}/{len(csv_files)} files")
        except Exception as e:
            logger.warning(f"  Error loading {csv_file.name}: {e}")

    if not all_dfs:
        logger.error("Failed to load any CSV files")
        return None

    # Concatenate all long-format data
    combined = pd.concat(all_dfs, ignore_index=True)
    del all_dfs
    gc.collect()
    logger.info(f"Combined long-format rows: {len(combined):,}")

    # PIVOT: long format → wide format (one column per metric)
    logger.info("Pivoting long → wide format...")
    pivoted = combined.pivot_table(
        index=['timestamp', 'cmdb_id'],
        columns='kpi_name',
        values='value',
        aggfunc='first'
    ).reset_index()
    pivoted.columns.name = None
    del combined
    gc.collect()

    # Assign new_container_id (alphabetically sorted cmdb_id → container_1..N)
    unique_cmdb = sorted(pivoted['cmdb_id'].unique())
    cmdb_to_id = {cmdb: f"container_{i+1}" for i, cmdb in enumerate(unique_cmdb)}
    pivoted['new_container_id'] = pivoted['cmdb_id'].map(cmdb_to_id)
    logger.info(f"Assigned IDs to {len(unique_cmdb)} unique containers")

    # Convert metrics to float32 to save RAM
    metric_cols = [c for c in pivoted.columns if c not in ['timestamp', 'cmdb_id', 'new_container_id']]
    for col in metric_cols:
        pivoted[col] = pd.to_numeric(pivoted[col], errors='coerce').astype(np.float32)

    # Sort by timestamp
    pivoted.sort_values(['timestamp', 'cmdb_id'], inplace=True)
    pivoted.reset_index(drop=True, inplace=True)

    logger.info(f"✓ Final shape: {pivoted.shape[0]:,} rows × {pivoted.shape[1]} columns")
    logger.info(f"  Columns: {list(pivoted.columns)}")
    return pivoted


df = load_data_from_drive_optimized(gd_raw_path)

if df is not None:
    print(f"✓ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Containers: {df['new_container_id'].nunique()}")
    log_memory("After loading data")
else:
    print("❌ Failed to load data")

# STEP 3a: Chronological Split + Normalize (Training Stats Only)

In [ ]:
if df is not None:
    logger.info("Splitting data chronologically (60/20/20)...")

    df.sort_values('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

    n = len(df)
    train_end = int(n * 0.6)
    val_end   = int(n * 0.8)

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    del df
    gc.collect()

    logger.info(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

    # Metric columns only (exclude ID/label columns)
    metric_cols = [
        c for c in train_df.columns
        if c not in ['timestamp', 'cmdb_id', 'new_container_id']
        and train_df[c].dtype in [np.float32, np.float64]
    ]

    # Calculate stats from TRAINING split only — no data leakage
    logger.info("Calculating normalization stats from training split only...")
    train_stats = {}
    for col in metric_cols:
        mean = float(train_df[col].mean())
        std  = float(train_df[col].std())
        train_stats[col] = {'mean': mean, 'std': std if std > 0 else 1.0}

    # Apply training stats to ALL splits
    def normalize_with_train_stats(data: pd.DataFrame, name: str) -> None:
        """Normalize in-place using TRAINING statistics only."""
        for col, s in train_stats.items():
            if col in data.columns:
                data[col] = ((data[col] - s['mean']) / s['std']).astype(np.float32)
        logger.info(f"  {name}: normalized {len(train_stats)} columns using training stats")

    normalize_with_train_stats(train_df, 'TRAIN')
    normalize_with_train_stats(val_df,   'VAL')
    normalize_with_train_stats(test_df,  'TEST')

    train_norm = train_df
    val_norm   = val_df
    test_norm  = test_df

    print("✓ Split and normalized (training stats only — no leakage)")
    print(f"  Train: {len(train_norm):,} rows | Val: {len(val_norm):,} | Test: {len(test_norm):,}")
    log_memory("After normalization")

# STEP 3b: Feature Engineering (Lag + Rolling per Container)

In [ ]:
if train_norm is not None:
    logger.info("=" * 70)
    logger.info("STEP 3b: FEATURE ENGINEERING")
    logger.info("=" * 70)

    target_columns = [
        'container_cpu_usage_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss'
    ]

    def engineer_features(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """
        Add lag diff and rolling features grouped by container.
        Uses groupby so features never bleed across container boundaries.
        """
        logger.info(f"Engineering features for {split_name}...")

        if 'new_container_id' not in data.columns:
            logger.warning(f"  {split_name}: no new_container_id — skipping feature engineering")
            return data

        available = [c for c in target_columns if c in data.columns]
        if not available:
            logger.warning(f"  {split_name}: no target columns found")
            return data

        data = data.sort_values(['new_container_id', 'timestamp']).reset_index(drop=True)

        grp = data.groupby('new_container_id')

        for col in available:
            # Lag difference features (within each container)
            for lag in [1, 2, 3]:
                data[f'{col}_DIFF_{lag}'] = (
                    grp[col].diff(lag).fillna(0).astype(np.float32)
                )
            # Rolling mean (within each container)
            data[f'{col}_ROLLING_MEAN_3'] = (
                grp[col]
                .transform(lambda x: x.rolling(3, min_periods=1).mean())
                .astype(np.float32)
            )
            # Rolling std (within each container)
            data[f'{col}_ROLLING_STD_3'] = (
                grp[col]
                .transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
                .astype(np.float32)
            )

        logger.info(f"  {split_name}: {len(data.columns)} total columns")
        return data

    train_feat = engineer_features(train_norm, 'TRAIN')
    val_feat   = engineer_features(val_norm,   'VAL')
    test_feat  = engineer_features(test_norm,  'TEST')

    # Save to disk — create merged_path only if needed
    merged_path.mkdir(parents=True, exist_ok=True)

    train_feat.to_csv(merged_path / 'train_data_with_features.csv', index=False)
    del train_feat, train_norm
    gc.collect()
    logger.info("  ✓ Train saved and cleared from RAM")

    val_feat.to_csv(merged_path / 'val_data_with_features.csv', index=False)
    del val_feat, val_norm
    gc.collect()
    logger.info("  ✓ Val saved and cleared from RAM")

    test_feat.to_csv(merged_path / 'test_data_with_features.csv', index=False)
    del test_feat, test_norm
    gc.collect()
    logger.info("  ✓ Test saved and cleared from RAM")

    print("✓ Feature engineering complete")
    print(f"  Files saved to: {merged_path}")
    log_memory("After feature engineering")

# STEP 4: Sliding Window Sequence Generation

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

class UltraFastSequenceGeneratorOptimized:
    """
    Fully vectorized sliding-window sequence generator.

    Key design:
      - sliding_window_view  : creates all X windows at once (no Python loop over positions)
      - Per-container files  : each container saves its own temp .npy (small, fast writes)
      - open_memmap merge    : final .npy written in a single O(n) pass
      - y_indices            : pre-computed once, not inside any loop
      Speedup vs loop-based: ~100x for large datasets
    """

    def __init__(self, lookback_window: int = 240, max_horizon: int = 10, batch_size: int = 2000):
        self.lookback_window = lookback_window
        self.max_horizon     = max_horizon
        self.batch_size      = batch_size   # kept for API compat, not used in vectorized path
        self.target_metrics  = [
            'container_cpu_usage_seconds_total',
            'container_memory_usage_bytes',
            'container_memory_working_set_bytes',
            'container_memory_rss'
        ]
        self.feature_cols = None

    def determine_feature_columns(self, df) -> list:
        exclude = {'timestamp', 'case_source', 'cmdb_id', 'new_container_id'}
        cols = [c for c in df.columns
                if c not in exclude
                and df[c].dtype in [np.float64, np.float32, int]]
        logger.info(f"Feature columns: {len(cols)}")
        return cols

    def process_and_save_sequences(self, csv_path: str, dataset_name: str, output_dir: str) -> bool:
        """
        Vectorized sequence generation:
          1. For each container: sliding_window_view creates all X windows in one numpy call
          2. NaN filtering done with boolean masks (no per-row Python checks)
          3. Each container writes its own small temp files
          4. Final merge uses open_memmap for O(n) disk writes
        """
        df            = None
        feature_data  = None
        container_ids = None

        try:
            logger.info(f"\nGenerating sequences for {dataset_name}...")
            df = pd.read_csv(csv_path)
            logger.info(f"  {len(df):,} rows x {len(df.columns)} columns")

            if self.feature_cols is None:
                self.feature_cols = self.determine_feature_columns(df)

            if 'new_container_id' not in df.columns or 'timestamp' not in df.columns:
                logger.error("Missing new_container_id or timestamp")
                return False

            df.sort_values(['new_container_id', 'timestamp'], inplace=True)
            df.reset_index(drop=True, inplace=True)

            output_path = Path(output_dir)
            output_path.mkdir(exist_ok=True)

            # Pre-compute y column indices once
            y_indices = [self.feature_cols.index(m)
                         for m in self.target_metrics if m in self.feature_cols]
            n_feat    = len(self.feature_cols)

            feature_data  = df[self.feature_cols].values.astype(np.float32)
            container_ids = df['new_container_id'].values

            del df
            df = None
            gc.collect()

            unique_containers = np.unique(container_ids)
            logger.info(f"Processing {len(unique_containers)} containers "
                        f"(lookback={self.lookback_window}, horizons=1-{self.max_horizon})...")

            # Track per-horizon counts and which containers contributed
            sequence_counts   = {h: 0 for h in range(1, self.max_horizon + 1)}
            container_indices = {h: [] for h in range(1, self.max_horizon + 1)}

            for idx, container_id in enumerate(unique_containers):
                if (idx + 1) % 5 == 0:
                    total = sum(sequence_counts.values())
                    logger.info(f"  Container {idx+1}/{len(unique_containers)} "
                                f"({total:,} sequences so far)")

                mask   = container_ids == container_id
                c_idx  = np.where(mask)[0]
                n_rows = len(c_idx)

                min_rows = self.lookback_window + self.max_horizon + 1
                if n_rows < min_rows:
                    continue

                cont_data = feature_data[c_idx].astype(np.float32)  # (n_rows, n_feat)

                # VECTORIZED: create all X windows at once using stride tricks
                # sliding_window_view output shape: (n_rows-lw+1, 1, lw, n_feat) -> squeeze -> (n_valid_w, lw, n_feat)
                n_windows = n_rows - self.lookback_window - self.max_horizon
                if n_windows <= 0:
                    del cont_data
                    continue

                X_windows = sliding_window_view(
                    cont_data, (self.lookback_window, n_feat)
                )[:n_windows, 0, :, :]  # view: (n_windows, lookback, n_feat)

                # VECTORIZED NaN filter on X windows
                nan_in_X   = np.isnan(X_windows).any(axis=(1, 2))   # (n_windows,)
                valid_pos  = np.where(~nan_in_X)[0]                  # positions with clean X

                if len(valid_pos) == 0:
                    del cont_data, X_windows
                    gc.collect()
                    continue

                # Copy only the valid windows (necessary to own the data)
                X_clean = np.ascontiguousarray(X_windows[valid_pos], dtype=np.float32)
                del X_windows, nan_in_X
                gc.collect()

                for horizon in range(1, self.max_horizon + 1):
                    # Target row index in cont_data for each valid position
                    tgt_rows   = valid_pos + self.lookback_window + horizon - 1
                    in_bounds  = tgt_rows < n_rows

                    X_h = X_clean[in_bounds]
                    y_h = cont_data[tgt_rows[in_bounds]][:, y_indices].astype(np.float32)

                    # NaN filter on y
                    nan_in_y = np.isnan(y_h).any(axis=1)
                    if nan_in_y.any():
                        X_h = X_h[~nan_in_y]
                        y_h = y_h[~nan_in_y]

                    if len(X_h) == 0:
                        del X_h, y_h
                        continue

                    c_h = np.full(len(X_h), container_id, dtype=object)

                    # Save this container slice as a temp file
                    np.save(str(output_path / f"_tmp_h{horizon}_X_{dataset_name}_{idx}.npy"), X_h)
                    np.save(str(output_path / f"_tmp_h{horizon}_y_{dataset_name}_{idx}.npy"), y_h)
                    np.save(str(output_path / f"_tmp_h{horizon}_c_{dataset_name}_{idx}.npy"), c_h)

                    sequence_counts[horizon]   += len(X_h)
                    container_indices[horizon].append(idx)

                    del X_h, y_h, c_h

                del cont_data, X_clean
                gc.collect()

            total = sum(sequence_counts.values())
            logger.info(f"Total valid sequences: {total:,}")

            # Merge per-container temp files into final .npy using memmap
            logger.info("Merging into final .npy files (memmap)...")
            n_containers_processed = len(unique_containers)

            for horizon in range(1, self.max_horizon + 1):
                if sequence_counts[horizon] == 0:
                    continue
                self._merge_container_files(
                    horizon, dataset_name, output_path,
                    container_indices[horizon], sequence_counts[horizon]
                )
                meta = {
                    'horizon':         horizon,
                    'dataset':         dataset_name,
                    'n_sequences':     sequence_counts[horizon],
                    'lookback_window': self.lookback_window,
                    'n_features':      len(self.feature_cols),
                    'feature_cols':    self.feature_cols,
                    'target_metrics':  self.target_metrics,
                    'X_shape':         [sequence_counts[horizon], self.lookback_window, len(self.feature_cols)],
                    'y_shape':         [sequence_counts[horizon], len(self.target_metrics)],
                }
                with open(output_path / f"sequences_horizon_{horizon}_metadata_{dataset_name}.json", 'w') as f:
                    json.dump(meta, f, indent=2)
                logger.info(f"  Horizon {horizon}: {sequence_counts[horizon]:,} sequences")

            return True

        except Exception as e:
            logger.error(f"Error processing {dataset_name}: {e}")
            import traceback
            logger.error(traceback.format_exc())
            return False

        finally:
            del df, feature_data, container_ids
            gc.collect()

    def _merge_container_files(self, horizon, dataset_name, output_path,
                                container_idx_list, n_sequences):
        """
        Merge per-container temp files into final .npy.
        Uses local /content/ for memmap because Google Drive FUSE
        does not support mmap for large file pre-allocation.
        Finished files are then moved to Drive.
        """
        import shutil
        n_feat    = len(self.feature_cols)
        n_targets = len(self.target_metrics)

        # Local temp paths — mmap works on local Colab disk
        local_tmp = Path('/content/_merge_tmp')
        local_tmp.mkdir(exist_ok=True)
        X_local = local_tmp / f"sequences_horizon_{horizon}_X_{dataset_name}.npy"
        y_local = local_tmp / f"sequences_horizon_{horizon}_y_{dataset_name}.npy"

        # Final destination on Drive
        X_final = output_path / f"sequences_horizon_{horizon}_X_{dataset_name}.npy"
        y_final = output_path / f"sequences_horizon_{horizon}_y_{dataset_name}.npy"
        c_path  = output_path / f"sequences_horizon_{horizon}_containers_{dataset_name}.npy"

        # Memmap write on local disk (avoids Drive FUSE mmap limitation)
        X_mm = np.lib.format.open_memmap(
            str(X_local), mode='w+', dtype=np.float32,
            shape=(n_sequences, self.lookback_window, n_feat))
        y_mm = np.lib.format.open_memmap(
            str(y_local), mode='w+', dtype=np.float32,
            shape=(n_sequences, n_targets))
        c_all = []

        offset = 0
        for c_idx in container_idx_list:
            X_tmp = output_path / f"_tmp_h{horizon}_X_{dataset_name}_{c_idx}.npy"
            y_tmp = output_path / f"_tmp_h{horizon}_y_{dataset_name}_{c_idx}.npy"
            c_tmp = output_path / f"_tmp_h{horizon}_c_{dataset_name}_{c_idx}.npy"

            if not X_tmp.exists():
                continue

            X_b = np.load(str(X_tmp))
            y_b = np.load(str(y_tmp))
            c_b = np.load(str(c_tmp), allow_pickle=True)
            n   = len(X_b)

            X_mm[offset:offset + n] = X_b
            y_mm[offset:offset + n] = y_b
            c_all.extend(c_b.tolist())
            offset += n

            del X_b, y_b, c_b
            X_tmp.unlink()
            y_tmp.unlink()
            c_tmp.unlink()
            gc.collect()

        # Flush and close memmap before moving
        del X_mm, y_mm
        gc.collect()

        # Move from local disk to Drive
        logger.info(f"    h{horizon}: moving merged files to Drive...")
        shutil.move(str(X_local), str(X_final))
        shutil.move(str(y_local), str(y_final))
        np.save(str(c_path), np.array(c_all, dtype=object))
        del c_all
        gc.collect()
        logger.info(f"    h{horizon}: merged {offset} sequences -> Drive")

    def diagnose(self, merged_path, sequences_path) -> None:
        merged_path    = Path(merged_path)
        sequences_path = Path(sequences_path)
        print("\n" + "="*70)
        print("DIAGNOSING SEQUENCE GENERATION")
        print("="*70)
        for name, f in [('train', merged_path / 'train_data_with_features.csv'),
                        ('val',   merged_path / 'val_data_with_features.csv'),
                        ('test',  merged_path / 'test_data_with_features.csv')]:
            print(f"  {name}: exists={f.exists()}")
            if f.exists():
                s = pd.read_csv(f, nrows=2)
                print(f"    cols={list(s.columns)[:5]} | has_id={'new_container_id' in s.columns}")
        print(f"  sequences dir exists: {sequences_path.exists()}")
        if sequences_path.exists():
            files = list(sequences_path.iterdir())
            print(f"  files: {len(files)}")
            for f in sorted(files)[:10]:
                print(f"    {f.name}")

print("✓ UltraFastSequenceGeneratorOptimized (vectorized) ready")

In [ ]:
logger.info("\n" + "#"*70)
logger.info("STEP 4: SEQUENCE GENERATION")
logger.info("#"*70)

generator = UltraFastSequenceGeneratorOptimized(lookback_window=240, max_horizon=10, batch_size=500)

datasets = [
    ('train', merged_path / 'train_data_with_features.csv'),
    ('val',   merged_path / 'val_data_with_features.csv'),
    ('test',  merged_path / 'test_data_with_features.csv'),
]

all_success = True
for data_name, csv_file in datasets:
    logger.info(f"\nProcessing {data_name.upper()}...")
    success = generator.process_and_save_sequences(str(csv_file), data_name, str(sequences_path))
    if not success:
        logger.error(f"❌ Failed to generate sequences for {data_name}")
        all_success = False
    log_memory(f"After {data_name} sequences")

if all_success:
    print("\n✓ All sequences generated successfully")
else:
    print("\n⚠ Some datasets failed — check logs above")

# STEP 7: Verify Sequences

In [ ]:
print("="*70)
print("VERIFYING SEQUENCES")
print("="*70)

try:
    X_train = np.load(sequences_path / 'sequences_horizon_1_X_train.npy')
    y_train = np.load(sequences_path / 'sequences_horizon_1_y_train.npy')

    print(f"\nTraining Data (Horizon 1):")
    print(f"  X shape: {X_train.shape}")
    print(f"  y shape: {y_train.shape}")

    npy_files = list(sequences_path.glob('*.npy'))
    json_files = list(sequences_path.glob('*.json'))

    print(f"\n  Total .npy files: {len(npy_files)}")
    print(f"  Total .json files: {len(json_files)}")

    print("\n" + "="*70)
    print("✅ SEQUENCES GENERATED SUCCESSFULLY!")
    print("="*70)

    final_mem = log_memory("Final")
    print(f"\n💾 Memory Summary:")
    print(f"  Initial: {initial_mem:.1f} MB")
    print(f"  Final:   {final_mem:.1f} MB")
    print(f"  Peak:    ~{final_mem:.1f} MB")

except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Sequences were not generated. Running diagnostics...\n")
    generator.diagnose(merged_path, sequences_path)

---

# Phase 2 — GRU Model Architecture & Data Pipeline

> **Prerequisites:** Run all Phase 1 cells above first.
> `sequences_path` is already defined and the `.npy` files are generated.

---

### What Phase 2 builds

| Component | Description |
|---|---|
| `SequenceDataset` | Custom Dataset wrapping (X, y) NumPy arrays |
| `load_dataset()` | Loads .npy files for a given horizon + split |
| `create_dataloader()` | Configures a DataLoader from arrays |
| `build_dataloaders()` | Builds train / val / test loaders in one call |
| `initialize_weights()` | Orthogonal (GRU) + Xavier (Linear) init |
| `GRUModel` | 2-layer GRU -> Dropout -> FC -> ReLU -> Output (batch, 4) |

## Step 2: Imports

In [ ]:
# Phase 2 additional imports (PyTorch — not needed in Phase 1)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"sequences_path  : {sequences_path}  (exists: {sequences_path.exists()})")

## Step 3: SequenceDataset

A custom `Dataset` that:
- Accepts raw NumPy arrays `(X, y)` from the Phase 1 `.npy` files
- Converts them to `float32` tensors **once** in `__init__` (not per-sample)
- Exposes `__len__` and `__getitem__` so PyTorch's DataLoader can index into it

In [ ]:
class SequenceDataset(Dataset):
    """
    PyTorch Dataset for pre-computed sliding-window sequences.

    Parameters
    ----------
    X : np.ndarray  shape (n_samples, seq_len, n_features)
        Input windows produced by the Phase 1 sequence generator.
        Example: (174220, 240, 27)
    y : np.ndarray  shape (n_samples, n_targets)
        Regression targets at the chosen prediction horizon.
        Example: (174220, 4)

    Notes
    -----
    Arrays are cast to float32 ONCE in __init__.
    No normalisation is applied here — sequences are already
    z-score normalised from Phase 1.
    """

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        if len(X) != len(y):
            raise ValueError(
                f"X and y must have the same number of samples "
                f"(got X={len(X)}, y={len(y)})"
            )
        # np.ascontiguousarray ensures tensors are in contiguous memory
        # which is required for efficient GPU transfer via pin_memory
        self.X = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))

    def __len__(self) -> int:
        """Total number of samples in the dataset."""
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Return one (input_window, target) pair.

        Returns
        -------
        x : torch.Tensor  shape (seq_len, n_features)  e.g. (240, 27)
        y : torch.Tensor  shape (n_targets,)            e.g. (4,)
        """
        return self.X[idx], self.y[idx]

    # ── Informational properties ──────────────────────────────────────
    @property
    def n_samples(self)  -> int: return len(self.X)
    @property
    def seq_len(self)    -> int: return self.X.shape[1]
    @property
    def n_features(self) -> int: return self.X.shape[2]
    @property
    def n_targets(self)  -> int: return self.y.shape[1]

    def __repr__(self) -> str:
        return (f"SequenceDataset("
                f"n_samples={self.n_samples}, "
                f"seq_len={self.seq_len}, "
                f"n_features={self.n_features}, "
                f"n_targets={self.n_targets})")

print("SequenceDataset defined.")

## Step 4: load_dataset()

Loads a single `(X, y)` pair of `.npy` files for a given **horizon** and **split**.

File naming convention (from Phase 1):
```
sequences_horizon_{horizon}_X_{split}.npy
sequences_horizon_{horizon}_y_{split}.npy
```

In [ ]:
def load_dataset(
    sequences_dir: str,
    horizon: int,
    split: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load pre-generated (X, y) NumPy files for one horizon and data split.

    Parameters
    ----------
    sequences_dir : str
        Path to the directory containing .npy files.
        In Colab: '/content/processed_data/sequences'
    horizon : int
        Prediction horizon, 1 to 10.
        horizon=1  -> predict 1 step ahead  (15 seconds)
        horizon=10 -> predict 10 steps ahead (2.5 minutes)
    split : str
        One of 'train', 'val', or 'test'.

    Returns
    -------
    X : np.ndarray  shape (n_samples, seq_len, n_features)
    y : np.ndarray  shape (n_samples, n_targets)

    Raises
    ------
    ValueError        if horizon not in [1,10] or split is unrecognised
    FileNotFoundError if .npy files are missing from sequences_dir
    """
    valid_splits = {'train', 'val', 'test'}
    if split not in valid_splits:
        raise ValueError(f"split must be one of {valid_splits}, got '{split}'")
    if not (1 <= horizon <= 10):
        raise ValueError(f"horizon must be 1-10, got {horizon}")

    base   = Path(sequences_dir)
    X_path = base / f"sequences_horizon_{horizon}_X_{split}.npy"
    y_path = base / f"sequences_horizon_{horizon}_y_{split}.npy"

    if not X_path.exists():
        raise FileNotFoundError(f"X file not found: {X_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"y file not found: {y_path}")

    X = np.load(str(X_path))
    y = np.load(str(y_path))

    logger.info(f"[load_dataset] horizon={horizon} {split} | X={X.shape} y={y.shape}")
    return X, y

print("load_dataset() defined.")

## Step 5: create_dataloader()

Wraps `(X, y)` NumPy arrays in a PyTorch `DataLoader` with configurable settings.

Key choices:
- `shuffle=True` for **training only** — prevents the model from memorising batch order
- `num_workers=0` — safest setting in Colab (multiprocessing is restricted in notebooks)
- `pin_memory=True` when a GPU is available — speeds up CPU to GPU tensor transfer

In [ ]:
def create_dataloader(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int  = 64,
    shuffle: bool    = True,
    num_workers: int = 0,
    pin_memory: bool = False,
    drop_last: bool  = False,
) -> DataLoader:
    """
    Build a DataLoader from NumPy arrays.

    Parameters
    ----------
    X, y        : NumPy arrays (see load_dataset for shapes)
    batch_size  : Samples per gradient update. Typical: 32, 64, 128.
    shuffle     : True for train, False for val/test.
    num_workers : Parallel loading workers. Use 0 in Colab.
    pin_memory  : Set True when training on GPU (faster CPU->GPU copy).
    drop_last   : Discard final incomplete batch. Useful with BatchNorm layers.

    Returns
    -------
    DataLoader that yields (X_batch, y_batch) tensors each iteration.
    """
    dataset = SequenceDataset(X, y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=drop_last,
    )

    logger.info(
        f"[create_dataloader] {len(dataset):,} samples | "
        f"{len(loader)} batches | batch_size={batch_size} | shuffle={shuffle}"
    )
    return loader

print("create_dataloader() defined.")

## Step 6: build_dataloaders()

Convenience function — builds all three DataLoaders (train / val / test) for a
given horizon in one call with the correct shuffle settings applied automatically.

In [ ]:
def build_dataloaders(
    sequences_dir: str,
    horizon: int,
    batch_size: int  = 64,
    num_workers: int = 0,
    pin_memory: bool = False,
) -> Dict[str, DataLoader]:
    """
    Load all three splits and return a dict of DataLoaders.

    Shuffle policy applied automatically:
        train -> shuffle=True
        val   -> shuffle=False
        test  -> shuffle=False

    Parameters
    ----------
    sequences_dir : str   Path to .npy files directory.
    horizon       : int   Prediction horizon (1-10).
    batch_size    : int   Shared batch size for all loaders.
    num_workers   : int   Parallel workers (0 for Colab).
    pin_memory    : bool  Pin tensors to CUDA memory when using GPU.

    Returns
    -------
    dict  {'train': DataLoader, 'val': DataLoader, 'test': DataLoader}

    Example
    -------
    loaders = build_dataloaders(str(sequences_path), horizon=1, batch_size=64)
    X_batch, y_batch = next(iter(loaders['train']))
    print(X_batch.shape)  # (64, 240, 27)
    print(y_batch.shape)  # (64, 4)
    """
    split_cfg = [
        ('train', True),
        ('val',   False),
        ('test',  False),
    ]

    loaders: Dict[str, DataLoader] = {}
    for split, shuffle in split_cfg:
        X, y = load_dataset(sequences_dir, horizon, split)
        loaders[split] = create_dataloader(
            X, y,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    return loaders

print("build_dataloaders() defined.")

## Step 7: initialize_weights()

Called via `model.apply(initialize_weights)` which visits **every sub-module**
in the model tree one at a time.

| Layer | Weight type | Method | Why |
|---|---|---|---|
| GRU | `weight_ih` (input to hidden) | Xavier uniform | Stable activation variance at input gate |
| GRU | `weight_hh` (hidden to hidden) | **Orthogonal** | Eigenvalues = 1, prevents gradient vanish/explode through time |
| GRU | biases | Zeros | Clean start |
| Linear | weights | Xavier uniform | Stable variance across FC layers |
| Linear | biases | Zeros | Clean start |

In [ ]:
def initialize_weights(module: nn.Module) -> None:
    """
    Best-practice weight initialisation for GRU + Linear layers.

    Called via:  model.apply(initialize_weights)
    PyTorch walks every sub-module and passes it here individually.

    Rules
    -----
    GRU weight_ih  -> Xavier uniform
        Keeps the variance of activations stable at the input gate.

    GRU weight_hh  -> Orthogonal initialisation
        Orthogonal matrices have unit-magnitude eigenvalues.
        Gradients neither explode nor vanish as they flow backward
        through the recurrent connections.
        This is the standard best practice for RNN weight init.

    GRU / Linear biases -> Zeros
        Neutral starting point; the model learns offsets from data.

    Linear weights -> Xavier uniform
        Keeps variance stable across the fully connected head layers.
    """
    if isinstance(module, nn.GRU):
        for name, param in module.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)

    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

print("initialize_weights() defined.")

## Step 8: GRUModel

```
Input  (batch, 240, 27)
    |
    v
GRU  (2 stacked layers, hidden=128, inter-layer dropout=0.2)
    |   only h_n[-1] is used — the top layer's final hidden state
    v   shape: (batch, 128)
Dropout (0.2)
    |
    v
Linear  128 -> 64
    |
    v
ReLU
    |
    v
Linear  64 -> 4
    |
    v
Output (batch, 4)
```

**Why `h_n[-1]` and not `gru_out[:, -1, :]`?**
Both are numerically identical. `h_n[-1]` is the explicit final hidden state
of the top GRU layer — the compressed temporal summary of the full 240-step window.

**Why no activation on the output layer?**
This is a regression task. Sigmoid/tanh would clip predictions.
MSELoss (Phase 3) expects raw unbounded values.

In [ ]:
class GRUModel(nn.Module):
    """
    Stacked GRU network for multi-step container resource forecasting.

    One independent instance is trained per prediction horizon.
    horizon=1  -> model predicts 1 timestep  (15 sec) ahead
    horizon=10 -> model predicts 10 timesteps (2.5 min) ahead

    Parameters
    ----------
    input_size  : int   Features per timestep. Default 27.
    hidden_size : int   GRU hidden units per layer. Default 128.
    num_layers  : int   Stacked GRU layers. Default 2.
    output_size : int   Regression targets. Default 4.
    dropout     : float Dropout probability applied between GRU layers
                        and before the FC head. Range [0, 1).
    """

    def __init__(
        self,
        input_size:  int   = 27,
        hidden_size: int   = 128,
        num_layers:  int   = 2,
        output_size: int   = 4,
        dropout:     float = 0.2,
    ) -> None:
        super().__init__()

        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.output_size = output_size
        self.dropout_p   = dropout

        # ── GRU stack ────────────────────────────────────────────────────
        # batch_first=True  expects (batch, seq_len, features)
        #                   returns (batch, seq_len, hidden_size)
        # dropout is applied between stacked layers only
        # (PyTorch ignores dropout for single-layer GRUs)
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # ── Dropout before FC head ────────────────────────────────────────
        # Applied to the final hidden state.
        # Prevents co-adaptation of hidden units; reduces overfitting.
        self.dropout = nn.Dropout(p=dropout)

        # ── Fully connected head ──────────────────────────────────────────
        # Two-layer MLP compresses hidden_size -> 64 -> output_size.
        # The intermediate 64-unit layer learns non-linear combinations
        # of the GRU's compressed temporal representation.
        self.fc1  = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(64, output_size)

        # Apply weight initialisation to every sub-module in the model
        self.apply(initialize_weights)

        logger.info(
            f"[GRUModel] input={input_size} hidden={hidden_size} "
            f"layers={num_layers} output={output_size} dropout={dropout} | "
            f"params={self._count_params():,}"
        )

    # ── Forward pass ─────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : torch.Tensor  shape (batch_size, seq_len, input_size)
            A batch of input windows from the DataLoader.

        Returns
        -------
        torch.Tensor  shape (batch_size, output_size)
            Raw regression predictions (no output activation).

        Flow
        ----
        1. GRU processes the full 240-timestep sequence.
           gru_out : (batch, 240, hidden_size) -- all timestep outputs (unused)
           h_n     : (num_layers, batch, hidden_size) -- final hidden states

        2. h_n[-1] extracts the TOP layer's final hidden state: (batch, hidden_size)
           This is the most abstract temporal summary of the input window.

        3. Dropout -> FC1 -> ReLU -> FC2 maps it to 4 target predictions.
        """
        # Step 1: run GRU over the full sequence
        gru_out, h_n = self.gru(x)

        # Step 2: take only the top layer's final hidden state
        last_hidden = h_n[-1]            # (batch, hidden_size)

        # Step 3: FC head with dropout
        out = self.dropout(last_hidden)  # (batch, hidden_size)
        out = self.fc1(out)              # (batch, 64)
        out = self.relu(out)             # (batch, 64)
        out = self.fc2(out)              # (batch, output_size)

        return out

    # ── Utility methods ───────────────────────────────────────────────────

    def _count_params(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def summary(self) -> None:
        """Print a human-readable architecture summary."""
        sep = "=" * 58
        print(sep)
        print("GRUModel - Architecture Summary")
        print(sep)
        print(f"  Input size   : {self.input_size}  (features per timestep)")
        print(f"  Hidden size  : {self.hidden_size}")
        print(f"  GRU layers   : {self.num_layers}")
        print(f"  Dropout      : {self.dropout_p}")
        print(f"  Output size  : {self.output_size}  (regression targets)")
        print("-" * 58)
        print(f"  Layer           Output shape")
        print(f"  GRU (stack)     (batch, 240, {self.hidden_size}) -> h_n[-1]: (batch, {self.hidden_size})")
        print(f"  Dropout         (batch, {self.hidden_size})")
        print(f"  Linear  fc1     (batch, 64)")
        print(f"  ReLU            (batch, 64)")
        print(f"  Linear  fc2     (batch, {self.output_size})")
        print(sep)
        print(f"  Trainable parameters : {self._count_params():,}")
        print(sep)

    def __repr__(self) -> str:
        return (f"GRUModel(input={self.input_size}, hidden={self.hidden_size}, "
                f"layers={self.num_layers}, output={self.output_size}, "
                f"dropout={self.dropout_p}, params={self._count_params():,})")

print("GRUModel defined.")

## Step 9: Smoke Test

Verifies every component works correctly before connecting to real data.

**Checks:**
1. `GRUModel` forward pass produces correct output shape `(64, 4)`
2. `SequenceDataset` wraps arrays correctly and is indexable
3. `DataLoader` yields batches with the right shapes
4. `build_dataloaders()` loads real `.npy` files from the sequences directory

In [ ]:
print("=" * 58)
print("SMOKE TEST - Phase 2 Components")
print("=" * 58)

# ── Test 1: Model forward pass ────────────────────────────────
print("
[1] GRUModel forward pass")
model = GRUModel(input_size=27, hidden_size=128, num_layers=2,
                 output_size=4, dropout=0.2)
model.summary()
dummy_input = torch.randn(64, 240, 27)
dummy_out   = model(dummy_input)
print(f"    Input  shape : {tuple(dummy_input.shape)}")
print(f"    Output shape : {tuple(dummy_out.shape)}")
assert dummy_out.shape == (64, 4)
print("    PASSED")

# ── Test 2: SequenceDataset ───────────────────────────────────
print("
[2] SequenceDataset")
X_fake = np.random.randn(500, 240, 27).astype(np.float32)
y_fake = np.random.randn(500, 4).astype(np.float32)
ds = SequenceDataset(X_fake, y_fake)
print(f"    {ds}")
x0, y0 = ds[0]
assert x0.shape == (240, 27)
assert y0.shape == (4,)
print("    PASSED")

# ── Test 3: DataLoader ────────────────────────────────────────
print("
[3] create_dataloader")
loader = create_dataloader(X_fake, y_fake, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print(f"    Batch X : {tuple(xb.shape)}")
print(f"    Batch y : {tuple(yb.shape)}")
assert xb.shape == (64, 240, 27)
assert yb.shape == (64, 4)
print("    PASSED")

# ── Test 4: Real .npy files (mmap — no full file load) ───────
print("
[4] build_dataloaders (real .npy files, horizon=1)")
import os

if not sequences_path.exists():
    print(f"    SKIPPED - path not found: {sequences_path}")
else:
    # Step 1: Check file sizes (fast — no data read)
    print("    File size check:")
    all_ok = True
    for split in ['train', 'val', 'test']:
        X_f = sequences_path / f"sequences_horizon_1_X_{split}.npy"
        y_f = sequences_path / f"sequences_horizon_1_y_{split}.npy"
        x_mb = os.path.getsize(X_f) // (1024*1024) if X_f.exists() else 0
        y_kb = os.path.getsize(y_f) // 1024        if y_f.exists() else 0
        ok   = x_mb > 0 and y_kb > 0
        print(f"      {split:5s}  X={x_mb:,}MB  y={y_kb:,}KB  [{'OK' if ok else 'EMPTY/MISSING'}]")
        if not ok:
            all_ok = False

    if not all_ok:
        print("    Some files are empty or missing. Re-run Phase 1.")
    else:
        # Step 2: Load with mmap_mode='r' — reads only one batch, not the full file
        print("    Loading one batch per split using mmap (fast)...")
        passed = True
        for split in ['train', 'val', 'test']:
            X_f = sequences_path / f"sequences_horizon_1_X_{split}.npy"
            y_f = sequences_path / f"sequences_horizon_1_y_{split}.npy"
            try:
                X_mm = np.load(str(X_f), mmap_mode='r')   # memory-mapped, no full load
                y_mm = np.load(str(y_f), mmap_mode='r')
                n    = len(X_mm)
                # Read only first 64 rows (one batch) to verify data integrity
                X_batch = torch.from_numpy(np.array(X_mm[:64], dtype=np.float32))
                y_batch = torch.from_numpy(np.array(y_mm[:64], dtype=np.float32))
                n_batches = int(np.ceil(n / 64))
                print(f"      {split:5s}  n={n:,}  X={tuple(X_batch.shape)}  y={tuple(y_batch.shape)}  batches={n_batches}")
                assert X_batch.shape == (64, 240, 27), f"Wrong X shape: {X_batch.shape}"
                assert y_batch.shape == (64, 4),       f"Wrong y shape: {y_batch.shape}"
                del X_mm, y_mm, X_batch, y_batch
            except Exception as e:
                print(f"      {split:5s}  FAILED: {e}")
                passed = False
        print("    PASSED" if passed else "    FAILED — re-run Phase 1")

print("
" + "=" * 58)
print("ALL SMOKE TESTS PASSED")
print("=" * 58)
print("
Phase 2 complete. Waiting for approval to proceed to Phase 3.")

---

# Phase 3 — Training Loop

> **Prerequisites:** Phase 1 and Phase 2 cells must be run first.

### What Phase 3 builds

| Component | Description |
|---|---|
| `TrainingConfig` | Single dataclass holding all hyperparameters |
| `EarlyStopping` | Stops training when val loss stops improving |
| `train_one_epoch()` | One training pass with AMP + gradient clipping |
| `validate()` | Validation pass returning loss + MAE + RMSE per metric |
| `train_model()` | Full training loop for one horizon |
| `train_all_horizons()` | Trains 10 independent models, one per horizon |

### Best practices applied
- **AdamW** — decoupled weight decay, better generalization than Adam
- **ReduceLROnPlateau** — halves LR when val loss plateaus
- **Gradient clipping** — prevents exploding gradients in RNNs
- **Mixed precision (AMP)** — faster training, less GPU memory
- **Early stopping** — avoids overfitting, saves compute
- **Best checkpoint** — saves only the model with lowest val loss
- **Per-metric metrics** — MAE + RMSE for each of the 4 target metrics

## Phase 3 — Definitions

Run this cell once after every runtime restart to define all training functions.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import random
import numpy as np
from dataclasses import dataclass, field
from typing import List

# Self-contained setup — safe to run after any kernel restart
device           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sequences_path   = Path('/content/drive/My Drive/processed_data/sequences')
models_save_path = Path('/content/drive/My Drive/processed_data/models')
models_save_path.mkdir(parents=True, exist_ok=True)
print(f'Device          : {device}')
print(f'sequences_path  : {sequences_path}')
print(f'models_save_path: {models_save_path}')



def set_seed(seed: int = 42) -> None:
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)


@dataclass
class TrainingConfig:
    """
    Single source of truth for all training hyperparameters.
    Change values here — nothing else needs to be edited.

    Model
    -----
    input_size  : Features per timestep (must match Phase 1 output = 27)
    hidden_size : GRU hidden units. Larger = more capacity, more RAM.
    num_layers  : Stacked GRU depth.
    dropout     : Regularisation probability.
    output_size : Regression targets (must match Phase 1 output = 4).

    Training
    --------
    batch_size  : Sequences per gradient update.
    epochs      : Maximum training epochs (early stopping may stop sooner).
    lr          : Initial learning rate for AdamW.
    weight_decay: L2 regularisation strength (AdamW decoupled).
    grad_clip   : Max gradient norm. Critical for RNN stability (use 1.0).

    Scheduler (ReduceLROnPlateau)
    -----------------------------
    lr_patience : Epochs with no val improvement before LR is reduced.
    lr_factor   : Multiply LR by this when plateau detected (0.5 = halve).
    lr_min      : LR floor — never drops below this value.

    Early Stopping
    --------------
    es_patience : Epochs with no val improvement before training stops.
    es_min_delta: Minimum improvement to count as 'better' val loss.

    Horizons
    --------
    horizons    : Which horizons to train. Default = all 10.
    """
    # Model
    input_size:   int   = 27
    hidden_size:  int   = 128
    num_layers:   int   = 2
    dropout:      float = 0.2
    output_size:  int   = 4

    # Training
    batch_size:   int   = 64
    epochs:       int   = 50
    lr:           float = 1e-3
    weight_decay: float = 1e-4
    grad_clip:    float = 1.0

    # LR Scheduler
    lr_patience:  int   = 3
    lr_factor:    float = 0.5
    lr_min:       float = 1e-6

    # Early Stopping
    es_patience:  int   = 7
    es_min_delta: float = 1e-5

    # Horizons to train
    horizons: List[int] = field(default_factory=lambda: list(range(1, 11)))


cfg = TrainingConfig()

# Paths
models_save_path = Path('/content/drive/My Drive/processed_data/models')
models_save_path.mkdir(parents=True, exist_ok=True)

print('TrainingConfig:')
print(f'  Model       : input={cfg.input_size}, hidden={cfg.hidden_size}, layers={cfg.num_layers}, dropout={cfg.dropout}')
print(f'  Training    : batch={cfg.batch_size}, epochs={cfg.epochs}, lr={cfg.lr}, weight_decay={cfg.weight_decay}')
print(f'  Grad clip   : {cfg.grad_clip}')
print(f'  Scheduler   : patience={cfg.lr_patience}, factor={cfg.lr_factor}, min_lr={cfg.lr_min}')
print(f'  Early stop  : patience={cfg.es_patience}, min_delta={cfg.es_min_delta}')
print(f'  Horizons    : {cfg.horizons}')
print(f'  Device      : {device}')
print(f'  Models path : {models_save_path}')


class EarlyStopping:
    """
    Stops training when validation loss stops improving.

    Parameters
    ----------
    patience  : int   Epochs to wait after last improvement before stopping.
    min_delta : float Minimum change in val loss to count as improvement.
    """

    def __init__(self, patience: int = 7, min_delta: float = 1e-5) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.early_stop = False

    def step(self, val_loss: float) -> bool:
        """
        Call once per epoch with the current validation loss.

        Returns True if training should stop, False otherwise.
        """
        if val_loss < self.best_loss - self.min_delta:
            # Improvement found — reset counter
            self.best_loss = val_loss
            self.counter   = 0
        else:
            # No improvement
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

        return self.early_stop

    def __repr__(self) -> str:
        return (f'EarlyStopping(patience={self.patience}, '
                f'counter={self.counter}/{self.patience}, '
                f'best={self.best_loss:.6f})')

print('EarlyStopping defined.')
# == MmapSequenceDataset =======================================================
# SequenceDataset (Phase 2) loads the ENTIRE .npy file into RAM in __init__.
# For train X = 4.3 GB that OOMs Colab before training even starts.
# MmapSequenceDataset uses np.load(mmap_mode='r'):
#   - File is memory-mapped at open time (no data read)
#   - Each __getitem__ reads only the requested row from disk
#   - Peak RAM = one batch (~1.7 MB), not 4.3 GB
class MmapSequenceDataset(Dataset):
    def __init__(self, X_path: str, y_path: str) -> None:
        if not Path(X_path).exists():
            raise FileNotFoundError(f'X file not found: {X_path}')
        if not Path(y_path).exists():
            raise FileNotFoundError(f'y file not found: {y_path}')
        # mmap_mode='r' maps the file descriptor - nothing read into RAM yet
        self.X = np.load(X_path, mmap_mode='r')   # (n, seq_len, n_feat)
        self.y = np.load(y_path, mmap_mode='r')   # (n, n_targets)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        # np.array() materialises only this row (copies mmap slice)
        x = torch.from_numpy(np.array(self.X[idx], dtype=np.float32))
        y = torch.from_numpy(np.array(self.y[idx], dtype=np.float32))
        return x, y

    def __repr__(self) -> str:
        return (f'MmapSequenceDataset('
                f'n={len(self.X)}, '
                f'seq_len={self.X.shape[1]}, '
                f'n_feat={self.X.shape[2]})')


def build_mmap_dataloaders(
    sequences_dir: str,
    horizon: int,
    cfg: 'TrainingConfig',
) -> dict:
    '''Build train/val/test DataLoaders using memory-mapped arrays (no OOM).'''
    base = Path(sequences_dir)
    loaders = {}
    for split, shuffle in [('train', True), ('val', False), ('test', False)]:
        X_path = str(base / f'sequences_horizon_{horizon}_X_{split}.npy')
        y_path = str(base / f'sequences_horizon_{horizon}_y_{split}.npy')
        ds = MmapSequenceDataset(X_path, y_path)
        loaders[split] = DataLoader(
            ds,
            batch_size=cfg.batch_size,
            shuffle=shuffle,
            num_workers=0,
            pin_memory=device.type == 'cuda',
        )
        print(f'  [mmap] horizon={horizon} {split}: {len(ds):,} samples')
    return loaders

print('MmapSequenceDataset defined.')
print('build_mmap_dataloaders() defined.')




def train_one_epoch(
    model:     torch.nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: torch.nn.Module,
    scaler:    torch.cuda.amp.GradScaler,
    grad_clip: float,
) -> float:
    """
    One full training pass over the DataLoader.

    Parameters
    ----------
    model     : GRUModel in train() mode.
    loader    : Training DataLoader.
    optimizer : AdamW optimizer.
    criterion : MSELoss.
    scaler    : GradScaler for mixed precision (AMP).
    grad_clip : Maximum gradient norm (1.0 recommended for RNNs).

    Returns
    -------
    float : Mean training loss over all batches.
    """
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        # Faster than zero_grad() — frees gradient memory instead of zeroing
        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward pass
        # autocast runs computation in float16 on GPU (float32 on CPU)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            preds = model(X_batch)           # (batch, 4)
            loss  = criterion(preds, y_batch)

        # Backward with scaled loss (prevents float16 underflow)
        scaler.scale(loss).backward()

        # Unscale before clipping so clip threshold is in true gradient units
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        # Optimizer step + scaler update
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

print('train_one_epoch() defined.')


TARGET_NAMES = [
    'cpu_usage',
    'mem_usage',
    'mem_working_set',
    'mem_rss',
]

def validate(
    model:     torch.nn.Module,
    loader:    DataLoader,
    criterion: torch.nn.Module,
) -> dict:
    """
    Full validation pass — no gradients, no dropout.

    Parameters
    ----------
    model     : GRUModel in eval() mode.
    loader    : Validation or test DataLoader.
    criterion : MSELoss.

    Returns
    -------
    dict with keys:
        'loss'            : float  — mean MSE over all batches
        'mae_per_metric'  : list[float] — MAE for each of 4 targets
        'rmse_per_metric' : list[float] — RMSE for each of 4 targets
        'mae_mean'        : float  — mean MAE across all 4 targets
        'rmse_mean'       : float  — mean RMSE across all 4 targets
    """
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                preds = model(X_batch)
                loss  = criterion(preds, y_batch)

            total_loss  += loss.item()
            all_preds.append(preds.cpu().float())
            all_targets.append(y_batch.cpu().float())

    # Stack all predictions and targets
    all_preds   = torch.cat(all_preds,   dim=0)   # (n_samples, 4)
    all_targets = torch.cat(all_targets, dim=0)   # (n_samples, 4)

    # Per-metric MAE and RMSE
    abs_err  = (all_preds - all_targets).abs()    # (n_samples, 4)
    sq_err   = (all_preds - all_targets) ** 2     # (n_samples, 4)

    mae_per  = abs_err.mean(dim=0).tolist()        # [4]
    rmse_per = sq_err.mean(dim=0).sqrt().tolist()  # [4]

    return {
        'loss':            total_loss / len(loader),
        'mae_per_metric':  mae_per,
        'rmse_per_metric': rmse_per,
        'mae_mean':        float(sum(mae_per)  / len(mae_per)),
        'rmse_mean':       float(sum(rmse_per) / len(rmse_per)),
    }

print('validate() defined.')
print('Target metrics:', TARGET_NAMES)


def train_model(
    horizon:  int,
    loaders:  dict,
    cfg:      TrainingConfig,
    save_dir: Path,
) -> dict:
    """
    Train one GRUModel for the given prediction horizon.

    Parameters
    ----------
    horizon  : int            Prediction horizon (1-10).
    loaders  : dict           {'train': DataLoader, 'val': DataLoader, 'test': DataLoader}
    cfg      : TrainingConfig All hyperparameters.
    save_dir : Path           Directory to save best model checkpoint.

    Returns
    -------
    dict  Training history:
        'train_loss' : list[float]  Training loss per epoch
        'val_loss'   : list[float]  Val loss per epoch
        'val_mae'    : list[float]  Val mean MAE per epoch
        'val_rmse'   : list[float]  Val mean RMSE per epoch
        'best_epoch' : int          Epoch with best val loss
        'best_val_loss' : float     Best val loss achieved
    """
    print(f'\n{"-" * 58}')
    print(f'Training horizon {horizon:2d} | device={device}')
    print(f'{"-" * 58}')

    # ── Model, optimizer, criterion, scheduler ────────────────────────
    model = GRUModel(
        input_size=cfg.input_size,
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        output_size=cfg.output_size,
        dropout=cfg.dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    criterion = torch.nn.MSELoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',           # monitor val loss (lower = better)
        patience=cfg.lr_patience,
        factor=cfg.lr_factor,
        min_lr=cfg.lr_min,
    )

    # GradScaler for mixed precision — no-op on CPU
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')

    early_stopping = EarlyStopping(
        patience=cfg.es_patience,
        min_delta=cfg.es_min_delta,
    )

    # ── Training history ──────────────────────────────────────────────
    history = {
        'train_loss': [],
        'val_loss':   [],
        'val_mae':    [],
        'val_rmse':   [],
        'best_epoch': 0,
        'best_val_loss': float('inf'),
    }

    best_val_loss  = float('inf')
    checkpoint_path = save_dir / f'gru_horizon_{horizon:02d}_best.pt'

    # ── Epoch loop ────────────────────────────────────────────────────
    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(
            model, loaders['train'], optimizer, criterion, scaler, cfg.grad_clip
        )
        val_metrics = validate(model, loaders['val'], criterion)
        val_loss    = val_metrics['loss']

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_metrics['mae_mean'])
        history['val_rmse'].append(val_metrics['rmse_mean'])

        # Save best checkpoint
        if val_loss < best_val_loss - cfg.es_min_delta:
            best_val_loss          = val_loss
            history['best_epoch']  = epoch
            history['best_val_loss'] = val_loss
            torch.save({
                'epoch':       epoch,
                'horizon':     horizon,
                'model_state': model.state_dict(),
                'val_loss':    val_loss,
                'val_metrics': val_metrics,
                'config':      cfg,
            }, str(checkpoint_path))
            saved = '  [SAVED]'
        else:
            saved = ''

        print(
            f'  Epoch {epoch:3d}/{cfg.epochs} | '
            f'train={train_loss:.5f} | '
            f'val={val_loss:.5f} | '
            f'mae={val_metrics["mae_mean"]:.5f} | '
            f'lr={current_lr:.2e}'
            f'{saved}'
        )

        # Early stopping check
        if early_stopping.step(val_loss):
            print(f'  Early stopping at epoch {epoch} '
                  f'(no improvement for {cfg.es_patience} epochs)')
            break

    print(f'  Best val loss: {best_val_loss:.6f} at epoch {history["best_epoch"]}')
    print(f'  Checkpoint: {checkpoint_path.name}')

    del model, optimizer, scaler
    torch.cuda.empty_cache()

    return history

print('train_model() defined.')


def train_all_horizons(
    sequences_dir: str,
    cfg:           TrainingConfig,
    save_dir:      Path,
) -> dict:
    """
    Train one GRUModel per horizon and save results.

    Parameters
    ----------
    sequences_dir : str           Path to .npy files directory.
    cfg           : TrainingConfig Hyperparameters.
    save_dir      : Path          Where to save model checkpoints.

    Returns
    -------
    dict  {horizon: history_dict} for all trained horizons.
    """
    save_dir.mkdir(parents=True, exist_ok=True)
    all_histories = {}

    print('=' * 58)
    print(f'TRAINING {len(cfg.horizons)} GRU MODELS (one per horizon)')
    print(f'Sequences : {sequences_dir}')
    print(f'Checkpoints: {save_dir}')
    print('=' * 58)

    for horizon in cfg.horizons:
        # Build DataLoaders for this horizon only
        # (load lazily — don't hold all horizons in RAM at once)
        try:
            # build_mmap_dataloaders uses MmapSequenceDataset:
            # reads slices from disk, never loads full 4 GB file into RAM
            loaders = build_mmap_dataloaders(
                sequences_dir=sequences_dir,
                horizon=horizon,
                cfg=cfg,
            )
        except FileNotFoundError as e:
            print(f'  Horizon {horizon}: SKIPPED — {e}')
            continue

        history = train_model(horizon, loaders, cfg, save_dir)
        all_histories[horizon] = history

        # Free DataLoader memory before next horizon
        del loaders

    return all_histories

print('train_all_horizons() defined.')

## Phase 3 — Run Training

Trains one GRU model per horizon (1-10) and saves checkpoints to Google Drive.

In [ ]:
# Run training for all horizons
all_histories = train_all_horizons(
    sequences_dir=str(sequences_path),
    cfg=cfg,
    save_dir=models_save_path,
)

print('\n' + '=' * 58)
print('TRAINING COMPLETE')
print('=' * 58)

## Phase 3 — Training Summary

In [ ]:
print('=' * 58)
print('TRAINING SUMMARY — All Horizons')
print('=' * 58)
print(f'  {"Horizon":>8}  {"Best Epoch":>10}  {"Best Val Loss":>14}  {"Val MAE":>10}  {"Val RMSE":>10}')
print(f'  {"-"*8}  {"-"*10}  {"-"*14}  {"-"*10}  {"-"*10}')

for h, hist in sorted(all_histories.items()):
    best_ep  = hist['best_epoch']
    best_val = hist['best_val_loss']
    best_mae  = hist['val_mae'][best_ep - 1]
    best_rmse = hist['val_rmse'][best_ep - 1]
    print(f'  {h:>8}  {best_ep:>10}  {best_val:>14.6f}  {best_mae:>10.5f}  {best_rmse:>10.5f}')

# Show saved checkpoints
print(f'\nSaved checkpoints in: {models_save_path}')
ckpts = sorted(models_save_path.glob('gru_horizon_*.pt'))
for c in ckpts:
    size_mb = c.stat().st_size / (1024 * 1024)
    print(f'  {c.name}  ({size_mb:.1f} MB)')

print('\nPhase 3 complete. Waiting for approval to proceed to Phase 4.')